In [0]:

from pyspark.sql import functions as F

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

SILVER = "workspace.silver"
GOLD = "workspace.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

# Read the Silver tables
customers = spark.table(f"{SILVER}.customers")
products = spark.table(f"{SILVER}.products")
sellers = spark.table(f"{SILVER}.sellers")
orders = spark.table(f"{SILVER}.orders")
items = spark.table(f"{SILVER}.order_items")
payments = spark.table(f"{SILVER}.order_payments")
reviews = spark.table(f"{SILVER}.order_reviews")
translations = spark.table(
    f"{SILVER}.product_category_name_translation"
)

# --------------------------------------------------
# 2. DIMENSION TABLES
# --------------------------------------------------

# Customer dimension
dim_customers = customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
)

# Seller dimension
dim_sellers = sellers.select(
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
)

# Product dimension with English category translations
p = products.alias("p")
t = translations.alias("t")

dim_products = (
    p.join(
        t,
        F.col("p.product_category_name")
        == F.col("t.product_category_name"),
        "left",
    )
    .select(
        F.col("p.product_id"),
        F.col("p.product_category_name"),
        F.col("t.product_category_name_english"),
        F.col("p.product_name_lenght").alias(
            "product_name_length"
        ),
        F.col("p.product_description_lenght").alias(
            "product_description_length"
        ),
        F.col("p.product_photos_qty"),
        F.col("p.product_weight_g"),
        F.col("p.product_length_cm"),
        F.col("p.product_height_cm"),
        F.col("p.product_width_cm"),
    )
)

# Date dimension from order purchase dates
dim_date = (
    orders
    .filter(F.col("order_purchase_timestamp").isNotNull())
    .select(
        F.to_date("order_purchase_timestamp").alias(
            "calendar_date"
        )
    )
    .distinct()
    .withColumn(
        "date_key",
        F.date_format(
            "calendar_date", "yyyyMMdd"
        ).cast("int"),
    )
    .withColumn("year", F.year("calendar_date"))
    .withColumn("quarter", F.quarter("calendar_date"))
    .withColumn("month", F.month("calendar_date"))
    .withColumn(
        "month_name",
        F.date_format("calendar_date", "MMMM"),
    )
    .withColumn(
        "day_of_month",
        F.dayofmonth("calendar_date"),
    )
    .withColumn(
        "day_of_week",
        F.date_format("calendar_date", "EEEE"),
    )
)

# --------------------------------------------------
# 3. FACT TABLE: ORDERS
# Grain: one row per order
# --------------------------------------------------

fact_orders = (
    orders
    .withColumn(
        "order_date",
        F.to_date("order_purchase_timestamp"),
    )
    .withColumn(
        "order_date_key",
        F.date_format(
            "order_purchase_timestamp", "yyyyMMdd"
        ).cast("int"),
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            "order_delivered_customer_date",
            "order_purchase_timestamp",
        ),
    )
    .withColumn(
        "is_delivered_late",
        F.when(
            F.col("order_delivered_customer_date").isNotNull()
            & F.col("order_estimated_delivery_date").isNotNull(),
            F.col("order_delivered_customer_date")
            > F.col("order_estimated_delivery_date"),
        ).otherwise(F.lit(None).cast("boolean")),
    )
)

# --------------------------------------------------
# 4. FACT TABLE: ORDER ITEMS
# Grain: one row per order item
# --------------------------------------------------

oi = items.alias("oi")
o = orders.alias("o")
dp = dim_products.alias("dp")
ds = dim_sellers.alias("ds")

fact_order_items = (
    oi
    .join(
        o,
        F.col("oi.order_id") == F.col("o.order_id"),
        "left",
    )
    .join(
        dp,
        F.col("oi.product_id") == F.col("dp.product_id"),
        "left",
    )
    .join(
        ds,
        F.col("oi.seller_id") == F.col("ds.seller_id"),
        "left",
    )
    .select(
        F.col("oi.order_id").alias("order_id"),
        F.col("oi.order_item_id").alias("order_item_id"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("oi.product_id").alias("product_id"),
        F.col("oi.seller_id").alias("seller_id"),
        F.col("o.order_status").alias("order_status"),
        F.col("o.order_purchase_timestamp").alias(
            "order_purchase_timestamp"
        ),
        F.to_date(
            F.col("o.order_purchase_timestamp")
        ).alias("order_date"),
        F.date_format(
            F.col("o.order_purchase_timestamp"),
            "yyyyMMdd",
        ).cast("int").alias("order_date_key"),
        F.col("dp.product_category_name"),
        F.col("dp.product_category_name_english"),
        F.col("ds.seller_state").alias("seller_state"),
        F.col("oi.shipping_limit_date"),
        F.col("oi.price").alias("price"),
        F.col("oi.freight_value").alias("freight_value"),
        (
            F.col("oi.price") + F.col("oi.freight_value")
        ).alias("item_total_amount"),
    )
)

# --------------------------------------------------
# 5. FACT TABLE: PAYMENTS
# Grain: one row per payment record
# --------------------------------------------------

pay = payments.alias("pay")
o = orders.alias("o")

fact_payments = (
    pay
    .join(
        o,
        F.col("pay.order_id") == F.col("o.order_id"),
        "left",
    )
    .select(
        F.col("pay.order_id").alias("order_id"),
        F.col("pay.payment_sequential").alias(
            "payment_sequential"
        ),
        F.col("o.customer_id").alias("customer_id"),
        F.col("o.order_status").alias("order_status"),
        F.col("o.order_purchase_timestamp").alias(
            "order_purchase_timestamp"
        ),
        F.date_format(
            F.col("o.order_purchase_timestamp"),
            "yyyyMMdd",
        ).cast("int").alias("order_date_key"),
        F.col("pay.payment_type").alias("payment_type"),
        F.col("pay.payment_installments").alias(
            "payment_installments"
        ),
        F.col("pay.payment_value").alias("payment_value"),
    )
)

# --------------------------------------------------
# 6. FACT TABLE: REVIEWS
# Grain: one row per source review record
# --------------------------------------------------

r = reviews.alias("r")
o = orders.alias("o")

fact_reviews = (
    r
    .join(
        o,
        F.col("r.order_id") == F.col("o.order_id"),
        "left",
    )
    .select(
        F.col("r.review_id").alias("review_id"),
        F.col("r.order_id").alias("order_id"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("o.order_status").alias("order_status"),
        F.col("o.order_purchase_timestamp").alias(
            "order_purchase_timestamp"
        ),
        F.col("r.review_score").alias("review_score"),
        F.col("r.review_comment_title").alias(
            "review_comment_title"
        ),
        F.col("r.review_comment_message").alias(
            "review_comment_message"
        ),
        F.col("r.review_creation_date").alias(
            "review_creation_date"
        ),
        F.col("r.review_answer_timestamp").alias(
            "review_answer_timestamp"
        ),
    )
)

# --------------------------------------------------
# 7. WRITE GOLD DELTA TABLES
# --------------------------------------------------

gold_tables = {
    "dim_customers": dim_customers,
    "dim_products": dim_products,
    "dim_sellers": dim_sellers,
    "dim_date": dim_date,
    "fact_orders": fact_orders,
    "fact_order_items": fact_order_items,
    "fact_payments": fact_payments,
    "fact_reviews": fact_reviews,
}

summary = []

for table_name, dataframe in gold_tables.items():
    full_name = f"{GOLD}.{table_name}"

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )

    row_count = spark.table(full_name).count()

    summary.append((table_name, row_count))

    print(f"SUCCESS: {full_name} | {row_count:,} rows")

print("\nGOLD LAYER COMPLETED")

summary_df = spark.createDataFrame(
    summary,
    ["table_name", "row_count"],
)

display(summary_df.orderBy("table_name"))


SUCCESS: workspace.gold.dim_customers | 99,441 rows
SUCCESS: workspace.gold.dim_products | 32,951 rows
SUCCESS: workspace.gold.dim_sellers | 3,095 rows
SUCCESS: workspace.gold.dim_date | 634 rows
SUCCESS: workspace.gold.fact_orders | 99,441 rows
SUCCESS: workspace.gold.fact_order_items | 112,650 rows
SUCCESS: workspace.gold.fact_payments | 103,886 rows
SUCCESS: workspace.gold.fact_reviews | 99,224 rows

GOLD LAYER COMPLETED


table_name,row_count
dim_customers,99441
dim_date,634
dim_products,32951
dim_sellers,3095
fact_order_items,112650
fact_orders,99441
fact_payments,103886
fact_reviews,99224


In [0]:
%sql
SHOW TABLES IN workspace.gold;

database,tableName,isTemporary
gold,dim_customers,false
gold,dim_date,false
gold,dim_products,false
gold,dim_sellers,false
gold,fact_order_items,false
gold,fact_orders,false
gold,fact_payments,false
gold,fact_reviews,false


In [0]:
%sql

SELECT 'dim_customers' AS table_name, COUNT(*) AS row_count
FROM workspace.gold.dim_customers

UNION ALL

SELECT 'dim_products', COUNT(*)
FROM workspace.gold.dim_products

UNION ALL

SELECT 'dim_sellers', COUNT(*)
FROM workspace.gold.dim_sellers

UNION ALL

SELECT 'dim_date', COUNT(*)
FROM workspace.gold.dim_date

UNION ALL

SELECT 'fact_orders', COUNT(*)
FROM workspace.gold.fact_orders

UNION ALL

SELECT 'fact_order_items', COUNT(*)
FROM workspace.gold.fact_order_items

UNION ALL

SELECT 'fact_payments', COUNT(*)
FROM workspace.gold.fact_payments

UNION ALL

SELECT 'fact_reviews', COUNT(*)
FROM workspace.gold.fact_reviews

ORDER BY table_name;

table_name,row_count
dim_customers,99441
dim_date,634
dim_products,32951
dim_sellers,3095
fact_order_items,112650
fact_orders,99441
fact_payments,103886
fact_reviews,99224


In [0]:

%sql

SELECT
    'dim_customers.customer_id' AS check_name,
    COUNT(*) AS duplicate_key_groups
FROM (
    SELECT customer_id
    FROM workspace.gold.dim_customers
    GROUP BY customer_id
    HAVING customer_id IS NULL OR COUNT(*) > 1
) q

UNION ALL

SELECT
    'dim_products.product_id',
    COUNT(*)
FROM (
    SELECT product_id
    FROM workspace.gold.dim_products
    GROUP BY product_id
    HAVING product_id IS NULL OR COUNT(*) > 1
) q

UNION ALL

SELECT
    'dim_sellers.seller_id',
    COUNT(*)
FROM (
    SELECT seller_id
    FROM workspace.gold.dim_sellers
    GROUP BY seller_id
    HAVING seller_id IS NULL OR COUNT(*) > 1
) q

UNION ALL

SELECT
    'fact_orders.order_id',
    COUNT(*)
FROM (
    SELECT order_id
    FROM workspace.gold.fact_orders
    GROUP BY order_id
    HAVING order_id IS NULL OR COUNT(*) > 1
) q

UNION ALL

SELECT
    'fact_order_items (order_id, order_item_id)',
    COUNT(*)
FROM (
    SELECT order_id, order_item_id
    FROM workspace.gold.fact_order_items
    GROUP BY order_id, order_item_id
    HAVING order_id IS NULL
        OR order_item_id IS NULL
        OR COUNT(*) > 1
) q

UNION ALL

SELECT
    'fact_payments (order_id, payment_sequential)',
    COUNT(*)
FROM (
    SELECT order_id, payment_sequential
    FROM workspace.gold.fact_payments
    GROUP BY order_id, payment_sequential
    HAVING order_id IS NULL
        OR payment_sequential IS NULL
        OR COUNT(*) > 1
) q

ORDER BY check_name;


check_name,duplicate_key_groups
dim_customers.customer_id,0
dim_products.product_id,0
dim_sellers.seller_id,0
"fact_order_items (order_id, order_item_id)",0
fact_orders.order_id,0
"fact_payments (order_id, payment_sequential)",0


In [0]:

%sql

SELECT
    'fact_orders -> dim_customers' AS relationship,
    SUM(CASE WHEN f.customer_id IS NULL THEN 1 ELSE 0 END)
        AS null_keys,
    SUM(CASE
        WHEN f.customer_id IS NOT NULL
         AND d.customer_id IS NULL THEN 1
        ELSE 0
    END) AS orphan_keys
FROM workspace.gold.fact_orders f
LEFT JOIN workspace.gold.dim_customers d
    ON f.customer_id = d.customer_id

UNION ALL

SELECT
    'fact_order_items -> fact_orders',
    SUM(CASE WHEN f.order_id IS NULL THEN 1 ELSE 0 END),
    SUM(CASE
        WHEN f.order_id IS NOT NULL
         AND o.order_id IS NULL THEN 1
        ELSE 0
    END)
FROM workspace.gold.fact_order_items f
LEFT JOIN workspace.gold.fact_orders o
    ON f.order_id = o.order_id

UNION ALL

SELECT
    'fact_order_items -> dim_products',
    SUM(CASE WHEN f.product_id IS NULL THEN 1 ELSE 0 END),
    SUM(CASE
        WHEN f.product_id IS NOT NULL
         AND p.product_id IS NULL THEN 1
        ELSE 0
    END)
FROM workspace.gold.fact_order_items f
LEFT JOIN workspace.gold.dim_products p
    ON f.product_id = p.product_id

UNION ALL

SELECT
    'fact_order_items -> dim_sellers',
    SUM(CASE WHEN f.seller_id IS NULL THEN 1 ELSE 0 END),
    SUM(CASE
        WHEN f.seller_id IS NOT NULL
         AND s.seller_id IS NULL THEN 1
        ELSE 0
    END)
FROM workspace.gold.fact_order_items f
LEFT JOIN workspace.gold.dim_sellers s
    ON f.seller_id = s.seller_id

UNION ALL

SELECT
    'fact_payments -> fact_orders',
    SUM(CASE WHEN f.order_id IS NULL THEN 1 ELSE 0 END),
    SUM(CASE
        WHEN f.order_id IS NOT NULL
         AND o.order_id IS NULL THEN 1
        ELSE 0
    END)
FROM workspace.gold.fact_payments f
LEFT JOIN workspace.gold.fact_orders o
    ON f.order_id = o.order_id

UNION ALL

SELECT
    'fact_payments -> dim_customers',
    SUM(CASE WHEN f.customer_id IS NULL THEN 1 ELSE 0 END),
    SUM(CASE
        WHEN f.customer_id IS NOT NULL
         AND d.customer_id IS NULL THEN 1
        ELSE 0
    END)
FROM workspace.gold.fact_payments f
LEFT JOIN workspace.gold.dim_customers d
    ON f.customer_id = d.customer_id

UNION ALL

SELECT
    'fact_reviews -> fact_orders',
    SUM(CASE WHEN f.order_id IS NULL THEN 1 ELSE 0 END),
    SUM(CASE
        WHEN f.order_id IS NOT NULL
         AND o.order_id IS NULL THEN 1
        ELSE 0
    END)
FROM workspace.gold.fact_reviews f
LEFT JOIN workspace.gold.fact_orders o
    ON f.order_id = o.order_id

UNION ALL

SELECT
    'fact_reviews -> dim_customers',
    SUM(CASE WHEN f.customer_id IS NULL THEN 1 ELSE 0 END),
    SUM(CASE
        WHEN f.customer_id IS NOT NULL
         AND d.customer_id IS NULL THEN 1
        ELSE 0
    END)
FROM workspace.gold.fact_reviews f
LEFT JOIN workspace.gold.dim_customers d
    ON f.customer_id = d.customer_id

ORDER BY relationship;


relationship,null_keys,orphan_keys
fact_order_items -> dim_products,0,0
fact_order_items -> dim_sellers,0,0
fact_order_items -> fact_orders,0,0
fact_orders -> dim_customers,0,0
fact_payments -> dim_customers,0,0
fact_payments -> fact_orders,0,0
fact_reviews -> dim_customers,0,0
fact_reviews -> fact_orders,0,0


In [0]:
%sql

SELECT
    DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM')
        AS order_month,

    COUNT(DISTINCT order_id)
        AS delivered_orders,

    COUNT(*)
        AS delivered_order_items,

    ROUND(SUM(price), 2)
        AS product_revenue,

    ROUND(SUM(freight_value), 2)
        AS freight_value,

    ROUND(SUM(price + freight_value), 2)
        AS total_product_plus_freight

FROM workspace.gold.fact_order_items

WHERE order_status = 'delivered'
  AND order_purchase_timestamp IS NOT NULL

GROUP BY DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM')

ORDER BY order_month;

order_month,delivered_orders,delivered_order_items,product_revenue,freight_value,total_product_plus_freight
2016-09,1,3,134.97,8.49,143.46
2016-10,265,313,40325.11,6165.55,46490.66
2016-12,1,1,10.9,8.72,19.62
2017-01,750,913,111798.36,15684.01,127482.37
2017-02,1653,1858,234223.4,37015.92,271239.32
2017-03,2546,2897,359198.85,55132.1,414330.95
2017-04,2303,2569,340669.68,50142.72,390812.4
2017-05,3546,4004,489338.25,77513.15,566851.4
2017-06,3135,3489,421923.37,68127.0,490050.37
2017-07,3872,4416,481604.52,84694.56,566299.08


In [0]:

%sql

WITH categorized_items AS (
    SELECT
        COALESCE(
            NULLIF(TRIM(product_category_name_english), ''),
            NULLIF(TRIM(product_category_name), ''),
            'Unknown'
        ) AS product_category,
        order_id,
        price,
        freight_value
    FROM workspace.gold.fact_order_items
    WHERE order_status = 'delivered'
)

SELECT
    product_category,
    COUNT(DISTINCT order_id) AS delivered_orders,
    COUNT(*) AS delivered_order_items,
    ROUND(SUM(price), 2) AS product_revenue,
    ROUND(SUM(freight_value), 2) AS freight_revenue,
    ROUND(SUM(price + freight_value), 2) AS total_revenue
FROM categorized_items
GROUP BY product_category
ORDER BY product_revenue DESC
LIMIT 10;

product_category,delivered_orders,delivered_order_items,product_revenue,freight_revenue,total_revenue
health_beauty,8647,9465,1233131.72,178957.81,1412089.53
watches_gifts,5495,5859,1166176.98,98156.14,1264333.12
bed_bath_table,9272,10953,1023434.76,201774.5,1225209.26
sports_leisure,7530,8431,954852.55,163404.36,1118256.91
computers_accessories,6530,7644,888724.61,143999.16,1032723.77
furniture_decor,6307,8160,711927.69,168402.23,880329.92
housewares,5743,6795,615628.69,142763.56,758392.25
cool_stuff,3559,3718,610204.1,81476.79,691680.89
auto,3810,4140,578966.65,90488.1,669454.75
toys,3804,4030,471286.48,75774.58,547061.06


In [0]:

%sql

SELECT
    payment_type,

    COUNT(*) AS payment_records,

    COUNT(DISTINCT order_id) AS orders_using_method,

    ROUND(SUM(payment_value), 2) AS total_payment_value,

    ROUND(AVG(payment_value), 2) AS avg_payment_record_value,

    ROUND(AVG(payment_installments), 2) AS avg_installments

FROM workspace.gold.fact_payments

WHERE payment_type IS NOT NULL

GROUP BY payment_type

ORDER BY total_payment_value DESC;


payment_type,payment_records,orders_using_method,total_payment_value,avg_payment_record_value,avg_installments
credit_card,76795,76505,1.254208419E7,163.32,3.51
boleto,19784,19784,2869361.27,145.03,1.0
voucher,5775,3866,379436.87,65.7,1.0
debit_card,1529,1528,217989.79,142.57,1.0
not_defined,3,3,0.0,0.0,1.0


In [0]:

%sql

SELECT
    review_score,

    COUNT(*) AS review_records,

    COUNT(DISTINCT order_id) AS distinct_orders,

    ROUND(AVG(review_score), 2) AS avg_score,

    ROUND(
        100.0 * COUNT(*)
        / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_reviews

FROM workspace.gold.fact_reviews

WHERE review_score IS NOT NULL

GROUP BY review_score

ORDER BY review_score DESC;


review_score,review_records,distinct_orders,avg_score,percentage_of_reviews
5,57328,57076,5.0,57.78
4,19142,19098,4.0,19.29
3,8179,8160,3.0,8.24
2,3151,3148,2.0,3.18
1,11424,11393,1.0,11.51


In [0]:

%sql

WITH delivered AS (
    SELECT
        DATE_FORMAT(
            order_purchase_timestamp,
            'yyyy-MM'
        ) AS order_month,

        order_id,
        delivery_days,
        is_delivered_late,

        CASE
            WHEN order_delivered_customer_date IS NOT NULL
             AND order_estimated_delivery_date IS NOT NULL
            THEN 1
            ELSE 0
        END AS has_delivery_comparison

    FROM workspace.gold.fact_orders
    WHERE order_status = 'delivered'
)

SELECT
    order_month,

    COUNT(*) AS delivered_orders,

    SUM(has_delivery_comparison)
        AS orders_with_delivery_comparison,

    SUM(
        CASE WHEN is_delivered_late = true
             THEN 1 ELSE 0 END
    ) AS late_orders,

    SUM(
        CASE WHEN is_delivered_late = false
             THEN 1 ELSE 0 END
    ) AS on_time_or_early_orders,

    ROUND(
        100.0 * SUM(
            CASE WHEN is_delivered_late = true
                 THEN 1 ELSE 0 END
        ) / NULLIF(SUM(has_delivery_comparison), 0),
        2
    ) AS late_delivery_percentage,

    ROUND(AVG(delivery_days), 2)
        AS avg_delivery_days

FROM delivered
GROUP BY order_month
ORDER BY order_month;


order_month,delivered_orders,orders_with_delivery_comparison,late_orders,on_time_or_early_orders,late_delivery_percentage,avg_delivery_days
2016-09,1,1,1,0,100.00,55.0
2016-10,265,265,3,262,1.13,19.63
2016-12,1,1,0,1,0.00,5.0
2017-01,750,750,23,727,3.07,12.75
2017-02,1653,1653,53,1600,3.21,13.3
2017-03,2546,2546,142,2404,5.58,13.04
2017-04,2303,2303,181,2122,7.86,15.02
2017-05,3546,3545,128,3417,3.61,11.43
2017-06,3135,3135,121,3014,3.86,12.01
2017-07,3872,3872,133,3739,3.43,11.49


In [0]:

%sql

SELECT
    COUNT(*) AS total_review_records,

    ROUND(AVG(review_score), 2)
        AS overall_average_score,

    ROUND(
        100.0 * SUM(
            CASE WHEN review_score >= 4 THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    ) AS positive_review_percentage,

    ROUND(
        100.0 * SUM(
            CASE WHEN review_score <= 2 THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    ) AS negative_review_percentage,

    SUM(
        CASE WHEN review_score = 5 THEN 1 ELSE 0 END
    ) AS five_star_reviews,

    SUM(
        CASE WHEN review_score = 1 THEN 1 ELSE 0 END
    ) AS one_star_reviews

FROM workspace.gold.fact_reviews

WHERE review_score IS NOT NULL;


total_review_records,overall_average_score,positive_review_percentage,negative_review_percentage,five_star_reviews,one_star_reviews
99224,4.09,77.07,14.69,57328,11424
